# QD Active Boundary Learning (Colab Runner)

This notebook runs the `latent-dynamics qd-active` CLI on Colab GPUs and saves artifacts/results.

Before running: set `OPENAI_API_KEY` in Colab Secrets (or env var).

In [2]:
REPO_URL = "https://github.com/ndalton12/latent-dynamics.git"
BRANCH = "main"
WORKDIR = "/content/latent-dynamics"
VENV_DIR = f"{WORKDIR}/.venv"
VENV_PYTHON = f"{VENV_DIR}/bin/python"

RUN_NAME = "qd_active_colab"
OUTPUT_ROOT = "/content/qd_runs"
OUTPUT_JSON = f"{OUTPUT_ROOT}/{RUN_NAME}_report.json"

# Start small first, then scale up.
LABEL_BUDGET = 64
WARM_START = 32
BATCH_SIZE = 8
CANDIDATE_POOL = 32


In [ ]:
import pathlib
import subprocess
import sys

HOST_PYTHON = sys.executable

def run(*args: str) -> None:
    print("+", " ".join(args))
    subprocess.run(args, check=True)

# Install uv in the host kernel interpreter.
run(HOST_PYTHON, "-m", "pip", "install", "-U", "uv")

if not pathlib.Path(WORKDIR).exists():
    run("git", "clone", "--depth", "1", REPO_URL, WORKDIR)

%cd {WORKDIR}
if BRANCH:
    run("git", "config", "pull.rebase", "true")
    run("git", "fetch", "origin", BRANCH, "--depth", "1")
    run("git", "pull", "origin", BRANCH)
    run("git", "checkout", BRANCH)

# Create an isolated venv to avoid Colab system package ABI conflicts.
venv_dir = pathlib.Path(VENV_DIR)
venv_python = str(pathlib.Path(VENV_PYTHON))
run(HOST_PYTHON, "-m", "uv", "venv", str(venv_dir), "--python", HOST_PYTHON)

# Install runtime deps and this repo into the venv interpreter.
run(
    HOST_PYTHON,
    "-m",
    "uv",
    "pip",
    "install",
    "--python",
    venv_python,
    "accelerate",
    "bitsandbytes",
    "datasets",
    "safetensors",
    "scikit-learn",
    "transformers",
    "typer",
    "flashlite",
    "ribs",
    "tqdm",
)
run(HOST_PYTHON, "-m", "uv", "pip", "install", "--python", venv_python, ".")

# Validate import in the venv (this is the interpreter used for the CLI run).
run(venv_python, "-c", "import latent_dynamics; print('latent_dynamics import OK')")
print("Venv interpreter:", venv_python)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 107.3 MB/s eta 0:00:00


NameError: name 'WORKDIR' is not defined

: 

In [ ]:
import os

try:
    from google.colab import userdata
    key = userdata.get("OPENAI_API_KEY")
    if key:
        os.environ["OPENAI_API_KEY"] = key
except Exception:
    pass

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set. Add it in Colab secrets or os.environ."
print("OPENAI_API_KEY detected.")


In [ ]:
import pathlib
import subprocess

PYTHON = str(pathlib.Path(VENV_PYTHON))
assert pathlib.Path(PYTHON).exists(), f"Missing venv python: {PYTHON}. Run setup cell first."

pathlib.Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
log_path = pathlib.Path(OUTPUT_ROOT) / f"{RUN_NAME}.log"

cmd = [
    PYTHON, "-m", "latent_dynamics.cli", "qd-active",
    "--model", "gemma3_4b",
    "--layer", "5",
    "--label-budget", str(LABEL_BUDGET),
    "--warm-start-labels", str(WARM_START),
    "--batch-size", str(BATCH_SIZE),
    "--candidate-pool-size", str(CANDIDATE_POOL),
    "--rollout-batch-size", "8",
    "--proxy-batch-size", "16",
    "--template-dir", "src/latent_dynamics/prompts/qd_active",
    "--output-root", OUTPUT_ROOT,
    "--output-json", OUTPUT_JSON,
    "--progress",
]

print("Running command:")
print(" ".join(cmd))

with log_path.open("w", encoding="utf-8") as f:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        f.write(line)
    rc = proc.wait()

if rc != 0:
    raise RuntimeError(f"CLI run failed with exit code {rc}. See log: {log_path}")

print(f"\nRun complete. Log saved to: {log_path}")


In [ ]:
import json
import pathlib
import shutil

report_path = pathlib.Path(OUTPUT_JSON)
assert report_path.exists(), f"Missing report: {report_path}"

report = json.loads(report_path.read_text())
run_dir = pathlib.Path(report["run_dir"])
zip_base = pathlib.Path(OUTPUT_ROOT) / f"{RUN_NAME}_artifacts"
zip_path = pathlib.Path(shutil.make_archive(str(zip_base), "zip", run_dir))

print("Report:", report_path)
print("Run dir:", run_dir)
print("Artifacts zip:", zip_path)
print("Final strategies:", list(report.get("evaluation", {}).get("strategies", {}).keys()))


In [ ]:
# Optional: copy outputs to Google Drive
# from google.colab import drive
# import shutil, pathlib
# drive.mount('/content/drive')
# DEST = pathlib.Path('/content/drive/MyDrive/latent_dynamics_runs')
# DEST.mkdir(parents=True, exist_ok=True)
# shutil.copy2(OUTPUT_JSON, DEST / pathlib.Path(OUTPUT_JSON).name)
# shutil.copy2(f"{OUTPUT_ROOT}/{RUN_NAME}.log", DEST / f"{RUN_NAME}.log")
# shutil.copy2(f"{OUTPUT_ROOT}/{RUN_NAME}_artifacts.zip", DEST / f"{RUN_NAME}_artifacts.zip")
